In [ ]:
import numpy as np
from aeon.classification import BaseClassifier
from aeon.transformations.collection.convolution_based import MultiRocket
from aeon.transformations.collection.convolution_based._hydra import HydraTransformer
from aeon.utils.validation import check_n_jobs
from aeon.transformations.collection.interval_based import QUANTTransformer
import numpy as np
import polars as pl
from aeon.classification.base import BaseClassifier
from aeon.classification.feature_based import (
    Catch22Classifier,
)
import os
from aeon.transformations.collection.convolution_based import Rocket
from aeon.datasets.tsc_datasets import univariate
from sklearn.base import clone
from aeon.classification.convolution_based import MultiRocketHydraClassifier
from aeon.classification.convolution_based import RocketClassifier
from sklearn.metrics import accuracy_score
from aeon.classification.interval_based import QUANTClassifier
from autotsc import utils, models
from tqdm import tqdm
from aeon.classification.feature_based import Catch22Classifier
from aeon.classification.interval_based import QUANTClassifier
from tqdm import tqdm
from aeon.benchmarking import resampling
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
write_dir = "experiments/shuffle_vs_no_shuffle"
os.makedirs(write_dir, exist_ok=True)

In [ ]:
def get_model(name, random_state=None):
    if name == "hydra-mr":
        return MultiRocketHydraClassifier(random_state=random_state, n_jobs=-1)
    #elif name == 'pf':
    #    return ProximityForest(random_state=random_state, n_jobs=-1)
    #elif name == 'quant':
    #    return QUANTClassifier(random_state=random_state)
    #elif name == 'rdst':
    #    return RDSTClassifier(random_state=random_state, n_jobs=-1)
    else:
        raise ValueError(f"Unknown model name: {name}")

In [ ]:
for dataset in tqdm(univariate[::20]):
    for split in ['shuffled', 'original']:
        for run in range(15):
            for model_name in ['hydra-mr']:
                stats = {
                    'dataset': dataset,
                    'split': split,
                    'run': run,
                    'model': model_name,
                }

                hash_val = pl.DataFrame([stats]).hash_rows(seed=42, seed_1=1, seed_2=2, seed_3=3).item()
                file = f"{write_dir}/{hash_val}.parquet"

                if os.path.exists(file):
                    print(f"Skipping: Dataset={dataset}, Run={run}, Model={model_name}")
                    continue
                else:
                    print(f"Processing: Dataset={dataset}, Run={run}, Model={model_name}")

                X_train, y_train, X_test, y_test = utils.load_dataset(dataset)
                if split == 'shuffled':
                    X_train, y_train, X_test, y_test = resampling.stratified_resample_data(X_train, y_train, X_test, y_test,random_state=run)

                # add also kfold val loss
                model = get_model(model_name, random_state=run)
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                acc = accuracy_score(y_test, y_pred)
                #print(f"Accuracy: {acc}")

                stats['test_accuracy'] = acc
        
                df_stat = pl.DataFrame([stats])
                df_stat.write_parquet(file, mkdir=True)



In [ ]:
df = pl.read_parquet(f"{write_dir}/*.parquet")
df.group_by('dataset', 'split').len().sort('dataset', 'split')

In [ ]:
df = pl.read_parquet(
    write_dir
).filter(
    pl.col("model") == "hydra-mr"
).pivot(
    values="test_accuracy",
    index=["dataset"],
    on="split",
    aggregate_function="mean",
).drop_nulls()
df

In [ ]:
sns.scatterplot(
    data=df.to_pandas(),
    x="original",
    y="shuffled",
)
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlabel("Accuracy (original split)")
plt.ylabel("Accuracy (shuffled split)")
plt.title("Hydra-MR: Original vs Shuffled Split")
